# Fine-tune GLiNER Azerbaijani (Option A) on Colab T4

Thin wrapper around `scripts/finetune_gliner_az.py`. The training script auto-resumes from the previous epoch pushed to the Hub, so if this session disconnects you can re-run this notebook to pick up where it left off.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Left sidebar → key icon (Secrets): add `HF_TOKEN` (a write-scoped HuggingFace token).
3. Make sure `{HF_USER}/azerbaijani-ner-final` dataset already exists on the Hub (produced by `scripts/merge_all_datasets.py` run locally).

In [ ]:
# 1. Install dependencies
!pip install -q gliner==0.2.26 datasets huggingface_hub accelerate python-dotenv

In [ ]:
# 2. Pull HF_TOKEN from Colab secrets and log in
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# Optional: set HF_USERNAME explicitly if you want to push under an org
# os.environ["HF_USERNAME"] = "your-org"

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

In [ ]:
# 3. Clone (or update) the repo. Safe to re-run — pulls latest on each invocation.
import pathlib
REPO = "https://github.com/w95/gliner-az.git"
REPO_DIR = pathlib.Path("/content/gliner-az")
if REPO_DIR.exists():
    print(f"{REPO_DIR} exists — pulling latest...")
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone {REPO} {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
# 4. Verify GPU
!nvidia-smi

In [ ]:
# 5. Train (auto-resumes from the last epoch pushed to the Hub if interrupted)
#    - 10 epochs total is the Option A target; the script checks how many are
#      already done and picks up from there.
#    - Per-epoch checkpoints push to {HF_USER}/gliner-azerbaijani-ner-v1 (private).
!python scripts/finetune_gliner_az.py \
  --epochs 10 \
  --batch-size 8 \
  --focal-gamma 2.5 \
  --data-repo azerbaijani-ner-final \
  --model-repo gliner-azerbaijani-ner-v1